### 1. Preparação do Ambiente e Importação de Bibliotecas
Nesta célula inicial, configuramos o ambiente de trabalho carregando as bibliotecas essenciais para a Ciência de Dados e Machine Learning:

* **Pandas e Numpy**: Fundamentais para a manipulação de matrizes e tabelas de dados biomecânicos.
* **Scikit-Learn**: Fornece as ferramentas de divisão de dados (`train_test_split`), normalização (`StandardScaler`) e as métricas de validação.
* **XGBoost e Random Forest**: Os algoritmos de inteligência artificial que serão treinados para identificar o risco de entorse.
* **OS e Time**: Bibliotecas de sistema para gestão de ficheiros e medição da eficiência (tempo de execução) dos modelos.

In [1]:
# ==============================================================================
# CÉLULA 1: IMPORTAÇÃO DE BIBLIOTECAS
# Responsabilidade: Carregar todas as ferramentas necessárias para o projeto.
# ==============================================================================
import pandas as pd
import numpy as np
import os
import time

# Ferramentas de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix

print("✅ Bibliotecas carregadas com sucesso!")

✅ Bibliotecas carregadas com sucesso!


### 2. Carregamento e Consolidação dos Dados Brutos
Nesta etapa, o sistema acede à diretoria externa `Data` para importar os conjuntos de dados biomecânicos provenientes dos três artigos científicos base. 

* **Caminhos Relativos**: Utilizamos a biblioteca `os` para garantir que o código seja portátil entre diferentes sistemas operativos (Windows/Mac/Linux).
* **Tratamento de Exceções**: Implementámos um bloco `try-except` para capturar erros de ficheiro não encontrado, garantindo que o programa não falhe abruptamente caso a pasta `Data` esteja ausente ou mal posicionada.
* **Validação Inicial**: O sistema confirma o sucesso do carregamento exibindo o volume total de amostras prontas para processamento.

In [2]:
# ==============================================================================
# CÉLULA 2: CARREGAMENTO DE DADOS
# Responsabilidade: Aceder à pasta 'Data' e carregar os datasets localmente.
# ==============================================================================
print("--- FASE 1: CARREGAMENTO ---")

# Caminho relativo (funciona em qualquer PC com a pasta Data ao lado do Notebook)
BASE_PATH = "../Data" 

try:
    # Construindo os caminhos de forma robusta
    df1 = pd.read_csv(os.path.join(BASE_PATH, "Base Artigo 1.csv"), sep=',')
    df2 = pd.read_csv(os.path.join(BASE_PATH, "Base Artigo 2.csv"), sep=',')
    df3 = pd.read_csv(os.path.join(BASE_PATH, "Base Artigo 3.csv"), sep=',')
    
    print("✅ Arquivos carregados com sucesso!")
    print(f"Total de amostras: {len(df1) + len(df2) + len(df3)}")
    load_successful = True
    
except FileNotFoundError as e:
    print(f"❌ ERRO: Não encontrei a pasta ou os ficheiros em: {BASE_PATH}")
    load_successful = False

--- FASE 1: CARREGAMENTO ---
✅ Arquivos carregados com sucesso!
Total de amostras: 181


### 3. Engenharia de Features e Pré-processamento de Dados
Esta fase é crucial para garantir a qualidade das previsões do modelo **PlaySafe4All**. O objetivo é transformar dados heterogéneos numa matriz numérica padronizada.

* **Consolidação e Seleção**: Unimos os diferentes datasets e filtramos apenas as variáveis biomecânicas e de exposição (Match/Training Time) que têm relevância clínica para a entorse.
* **Tratamento de Dados Omissos**: Utilizamos a **Mediana** para preencher valores em falta, garantindo que o modelo não seja enviesado por valores extremos (outliers).
* **Codificação e Binarização**: O alvo (*Target*) é convertido para um formato binário (0 para saudável, 1 para entorse), facilitando a interpretação pelos algoritmos de classificação.
* **Divisão Estratificada**: Dividimos os dados em Treino e Teste mantendo a proporção original de lesões (`stratify=y`), o que evita que o modelo aprenda apenas sobre atletas saudáveis.
* **Normalização (StandardScaler)**: Reescalamos todas as variáveis para que tenham média 0 e desvio padrão 1. Isto impede que variáveis com números grandes (como o tempo em minutos) dominem variáveis com números pequenos (como razões de força).

In [3]:
# ==============================================================================
# CÉLULA 3: PRÉ-PROCESSAMENTO (SIMPLIFICADA)
# Responsabilidade: Transformar dados brutos em matrizes de treino/teste.
# ==============================================================================

print("\n--- FASE 2: PREPARAÇÃO DE DADOS ---")

# 1. Unir as bases e remover colunas inúteis (como ID/Número)
df = pd.concat([df1, df2, df3], ignore_index=True)
if 'Numero' in df.columns: df.drop(columns=['Numero'], inplace=True)

# 2. Seleção de colunas relevantes e Alvo
TARGET = 'Entorse'
cols = [TARGET, 'T0_T1_Match_Time_exposure', 'T0_T1_Training_Time_exposure', 
        'T0SRTMax', 'T0TTestMin', 'T0SJmMax', 'T0Veli', 'T0RazaoAADto', 'T0RazaoAAEsq']

# Mantém apenas o que existe nos ficheiros e preenche vazios com a mediana
df = df[[c for c in cols if c in df.columns]].copy()
df = df.fillna(df.median(numeric_only=True))

# 3. Definição de X e Y (Ajustando Y para 0 e 1 se necessário)
y = df[TARGET].apply(lambda x: 1 if x > 1 else 0) # Garante formato binário 0/1
X = pd.get_dummies(df.drop(columns=[TARGET]), drop_first=True)

# 4. Divisão e Escalonamento (Standardization)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.75, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f"✅ Dados Prontos! Treino: {X_train_scaled.shape} | Teste: {X_test_scaled.shape}")


--- FASE 2: PREPARAÇÃO DE DADOS ---
✅ Dados Prontos! Treino: (45, 56) | Teste: (136, 56)


### 4. Construção e Treino do Modelo (XGBoost)
Nesta etapa, implementamos o algoritmo **XGBoost (Extreme Gradient Boosting)**, uma técnica de aprendizagem supervisionada baseada em árvores de decisão. 

* **Aprendizagem Sequencial**: Ao contrário de outros modelos, o XGBoost cria árvores de forma sequencial, onde cada nova árvore tenta corrigir os erros cometidos pelas anteriores.
* **Hiperparâmetros de Controlo**:
    * `n_estimators=100`: Define o número de árvores na "floresta" sequencial.
    * `max_depth=3`: Limita a profundidade das árvores para evitar que o modelo se torne demasiado complexo e sofra de *overfitting* (decorar os dados em vez de aprender padrões).
    * `learning_rate=0.1`: Controla a velocidade com que o modelo aprende, garantindo uma convergência estável.
* **Objetivo**: O modelo utiliza as variáveis biomecânicas normalizadas para mapear a probabilidade de um atleta sofrer uma entorse, focando-se na minimização da perda logarítmica (*logloss*).

In [4]:
# ==============================================================================
# CÉLULA 4: TREINO DO MODELO (XGBOOST)
# Responsabilidade: Treinar o algoritmo com os dados escalonados.
# ==============================================================================
print("--- FASE 3: TREINO DO XGBOOST ---")

# 1. Criar o modelo
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric='logloss'
)

# 2. Treinar usando X_train_scaled e y_train (vindos da Célula 3)
xgb_model.fit(X_train_scaled, y_train)

print("✅ Modelo 'xgb_model' treinado com sucesso!")

--- FASE 3: TREINO DO XGBOOST ---
✅ Modelo 'xgb_model' treinado com sucesso!


### 5. Avaliação de Desempenho e Segurança Biometral
Nesta fase final, submetemos o modelo **XGBoost** a um conjunto de dados de teste (que o algoritmo nunca viu durante o treino) para validar a sua eficácia no mundo real.

* **Métricas de Performance**:
    * **Accuracy**: Mede a taxa de acerto global do modelo.
    * **Recall (Sensibilidade)**: É a nossa métrica mais crítica. Indica a percentagem de lesões reais que o modelo conseguiu prever corretamente.
    * **F1-Score**: O equilíbrio entre a precisão e a sensibilidade, garantindo que o modelo é estável.
* **Matriz de Confusão**: Uma tabela que nos permite visualizar os acertos e, principalmente, os erros do modelo.
* **Falsos Negativos (FN)**: Representam atletas em risco que o sistema classificou como "Saudáveis". No projeto **PlaySafe4All**, o objetivo é manter este número o mais próximo de **zero** possível, garantindo que nenhuma entorse passe despercebida pela equipa médica.
* **Eficiência Temporal**: Medimos o tempo de execução em milissegundos para assegurar que o sistema pode fornecer alertas em tempo real durante um treino ou jogo.

In [5]:
# ==============================================================================
# CÉLULA 5: AVALIAÇÃO FINAL (XGBOOST)
# Responsabilidade: Validar o desempenho e a segurança do modelo.
# ==============================================================================

print("\n--- FASE 4: AVALIAÇÃO FINAL (XGBoost) ---")

start_time = time.perf_counter()

# 1. Previsões
y_pred = xgb_model.predict(X_test_scaled)

# 2. Métricas (y_test já está em 0/1 vindo da sua Célula 3)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

execution_time_ms = (time.perf_counter() - start_time) * 1000

# 3. Resultados
print(f"📊 Precisão (Accuracy): {accuracy:.4f}")
print(f"🎯 Recall (Sensibilidade): {recall:.4f}")
print(f"🧪 F1-Score: {f1:.4f}")
print(f"⏱️ Tempo de Execução: {execution_time_ms:.2f} ms")

print("\nMatriz de Confusão:")
print(cm)

# 4. Validação Crítica
FN = cm[1, 0]
print(f"\n❌ Falsos Negativos (Lesões não previstas): {FN}")

if FN <= 2:
    print("\n--- ✅ PROJETO VALIDADO PELO RECALL ---")
else:
    print(f"\n--- ⚠️ AVISO: FN={FN}. RECOMENDA-SE REAJUSTAR O MODELO ---")


--- FASE 4: AVALIAÇÃO FINAL (XGBoost) ---
📊 Precisão (Accuracy): 0.6985
🎯 Recall (Sensibilidade): 0.6933
🧪 F1-Score: 0.7172
⏱️ Tempo de Execução: 20.60 ms

Matriz de Confusão:
[[43 18]
 [23 52]]

❌ Falsos Negativos (Lesões não previstas): 23

--- ⚠️ AVISO: FN=23. RECOMENDA-SE REAJUSTAR O MODELO ---
